# EfficientNet Fine-Tuning — Paper Revision

EfficientNetB0 with last 50 layers fine-tuned per GLOO fold. No data leakage: model re-initialized
from ImageNet weights per fold, trained exclusively on train data, `validation_data` used for
monitoring only (no early stopping, fixed 20 epochs).

Added for revision: animal-level metrics, temporal analysis, bootstrap CIs, per-fold prediction saving.

## Method
1. Load full dataset from TFRecord (`full_ds_fixed_no_control.tfrecord`)
2. Convert dataset to numpy arrays (X_all, y_all, mouse_ids_all, days_all)
3. Leave-One-Group-Out (LOGO) CV: 10 folds (one per mouse)
4. Per fold:
   - Build fresh EfficientNetB0 from ImageNet weights (last 50 layers unfrozen)
   - Train for 20 epochs on X_train only
   - validation_data=(X_test, y_test) for monitoring only (no early stopping)
   - Predict on held-out mouse exams
5. Pool predictions and compute exam-level, animal-level, temporal, bootstrap CI metrics

## Requirements
Requires GPU (TF 2.10 + CUDA 11.x). Cannot run in CPU-only venv.

In [ ]:
# NOTE: Requires GPU (TF 2.10 + CUDA 11.x). Cannot run in CPU-only venv.
import os
import sys

# Set working directory to repo root so config and src are importable.
# The kernel may start from a different directory when run via nbconvert.
_candidate = os.path.abspath('.')
if not os.path.isfile(os.path.join(_candidate, 'config.py')):
    _search = _candidate
    for _ in range(5):
        _search = os.path.dirname(_search)
        if os.path.isfile(os.path.join(_search, 'config.py')):
            _candidate = _search
            break
os.chdir(_candidate)
sys.path.insert(0, _candidate)

print('Working directory:', os.getcwd())

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
print(tf.__version__)
print(tf.test.is_gpu_available())

In [ ]:
import warnings
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

import config
from tfrecordhandlerDL import TFRecordDataHandlerDL

warnings.filterwarnings('ignore')

print('TF version:', tf.__version__)

## Configuration

Output directory for paper revision DL fine-tuning results.

In [ ]:
# Output directory for this experiment (paper revision)
OUTPUT_DIR = config.OUTPUT_EXP_DL_FT
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
INPUT_DIR  = 'input/deep_learning'
BATCH_SIZE = 4     # smaller batch for fine-tuning (GPU memory)
N_EPOCHS   = 20    # fixed — no early stopping
SEED       = 42
N_BOOTSTRAP = 10_000

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print('Output directory:', OUTPUT_DIR)
print(f'Epochs: {N_EPOCHS} (fixed, no early stopping)')

## Data Loading

Load the full dataset from TFRecord (no control mice). Each batch yields
`(images, masks, labels, mouse_ids, days_of_study)`.

In [ ]:
def visualize_data(image_batch, mask_batch, num_samples=4):
    plt.figure(figsize=(10, num_samples * 2))
    for i in range(num_samples):
        plt.subplot(num_samples, 2, 2 * i + 1)
        plt.imshow(np.squeeze(image_batch[i]), cmap='gray')
        plt.title(f'Image {i}')
        plt.axis('off')
        plt.subplot(num_samples, 2, 2 * i + 2)
        plt.imshow(np.squeeze(mask_batch[i]), cmap='gray')
        plt.title(f'Mask {i}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
train_tfrecord = os.path.join(INPUT_DIR, 'full_ds_fixed_no_control.tfrecord')

train_ds_handler = TFRecordDataHandlerDL(train_tfrecord, batch_size=BATCH_SIZE, shuffle=False, augment=False)
train_ds = train_ds_handler.dataset
print(f'Number of samples: {train_ds_handler.length}')

images_tf, masks_tf, group_names, mouse_ids, days_of_study = next(iter(train_ds))
print(f'Images min: {images_tf.numpy().min():.4f}, max: {images_tf.numpy().max():.4f}')

visualize_data(images_tf, masks_tf, num_samples=4)

## Augmentation Helper (defined for reference)

Augmentation is applied via the dataset pipeline during training if needed.

In [ ]:
def augment_data(image, mask, label, mouse_ids, days_of_study):
    seed = tf.random.uniform([], maxval=10000, dtype=tf.int32)
    image = tf.image.stateless_random_flip_left_right(image, seed=[seed, 1])
    mask = tf.image.stateless_random_flip_left_right(mask, seed=[seed, 1])
    image = tf.image.stateless_random_brightness(image, max_delta=0.2, seed=[seed, 2])
    image = tf.image.stateless_random_contrast(image, lower=0.8, upper=1.2, seed=[seed, 3])
    image = tf.image.stateless_random_saturation(image, lower=0.8, upper=1.2, seed=[seed, 4])
    image = tf.image.stateless_random_hue(image, max_delta=0.05, seed=[seed, 5])
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask, label, mouse_ids, days_of_study

## Model Architecture — EfficientNetB0 with Partial Fine-Tuning

For each fold a **fresh** model is built from ImageNet weights. The last 50 layers are unfrozen
and trained per fold on that fold's training data. Earlier layers remain frozen to preserve
low-level feature representations.

**No data leakage:** each fold trains from scratch using only the training mice for that fold.

In [ ]:
def build_model():
    """
    Build EfficientNetB0 with last 50 layers unfrozen.
    Called fresh for each GLOO fold — model is re-initialized from ImageNet weights.
    """
    input_layer = layers.Input(shape=(256, 256, 3), name='input_layer')

    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_layer)
    # Freeze all except the last 50 layers
    for layer in base_model.layers[:-50]:
        layer.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    output_layer = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=input_layer, outputs=output_layer)
    return model


print('Available devices:')
print(tf.config.list_physical_devices('GPU'))

# Verify model structure
_test_model = build_model()
_test_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(f'Trainable layers (last 50 unfrozen): {sum(1 for l in _test_model.layers if l.trainable)}')
del _test_model
gc.collect()

## Extract Full Dataset to Numpy Arrays

Convert the TFRecord dataset into numpy arrays before the LOGO loop. This includes building
the 3-channel input (T2 image + mask overlay) and collecting mouse_id and day_of_study metadata.

In [ ]:
def extract_data(dataset):
    """
    Convert TFRecord dataset to numpy arrays.
    Builds 3-channel input: [T2_image, T2_image, mask] stacked along last axis.

    Returns:
        X_all:         (N, 256, 256, 3) float32 numpy array
        y_all:         (N,) int array of labels
        mouse_ids_all: (N,) str array of mouse IDs
        days_all:      (N,) int array of days_of_study
    """
    images_list, labels_list, mouse_ids_list, days_list = [], [], [], []

    for batch in dataset:
        x_batch, masks_b, y_batch, mouse_batch, day_batch = batch

        if x_batch.shape[-1] == 1:
            x_batch = tf.squeeze(x_batch, axis=-1)  # (batch, 256, 256)

        # Build 3-channel: T2 image (ch0, ch1) + mask (ch2)
        x_batch_rgb = tf.stack([
            x_batch,           # Channel 0: T2 MRI
            x_batch,           # Channel 1: T2 MRI (repeated)
            masks_b[..., 0],   # Channel 2: tumour mask
        ], axis=-1)  # (batch, 256, 256, 3)

        images_list.append(x_batch_rgb)
        labels_list.append(y_batch)
        mouse_ids_list.append(mouse_batch)
        days_list.append(day_batch)

    X_all         = tf.concat(images_list, axis=0).numpy()  # (N, 256, 256, 3)
    y_all         = tf.concat(labels_list, axis=0).numpy().squeeze()
    mouse_ids_all = tf.concat(mouse_ids_list, axis=0).numpy().squeeze()
    days_all      = tf.concat(days_list, axis=0).numpy().squeeze()

    # Decode bytes to str
    if isinstance(mouse_ids_all[0], bytes):
        mouse_ids_all = np.array([m.decode('utf-8') for m in mouse_ids_all])

    print(f'[INFO] Dataset shape: {X_all.shape}')
    return X_all, y_all, mouse_ids_all, days_all


# Rebatch at fine-tuning batch size
train_ds_loop = train_ds.unbatch().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

X_all, y_all, mouse_ids_all, days_all = extract_data(train_ds_loop)

print(f'\nX_all shape:      {X_all.shape}')
print(f'y_all shape:      {y_all.shape}')
print(f'Mouse IDs:        {np.unique(mouse_ids_all)}')
print(f'Days range:       {days_all.min()} – {days_all.max()}')
print(f'Label counts:     0={int((y_all==0).sum())}  1={int((y_all==1).sum())}')

## LOGO Cross-Validation — EfficientNetB0 Fine-Tuning Per Fold

For each fold (one mouse left out as test):
1. Build fresh EfficientNetB0 from ImageNet weights (last 50 layers unfrozen)
2. Compile with Adam (lr=1e-4) + binary cross-entropy
3. Train for 20 epochs on X_train only; `validation_data=(X_test, y_test)` for monitoring only
   — **no early stopping**, fixed epoch count ensures no data leakage from val loss
4. Predict on held-out mouse exams
5. Track mouse_id and day_of_study per prediction
6. Save per-fold `predictions.csv`

**Leakage note:** using `validation_data` purely for monitoring (no callbacks that select weights
or stop training based on val loss) is standard practice and does not constitute leakage.
The final weights are those from the last epoch, trained on training data only.

In [ ]:
logo = LeaveOneGroupOut()

# Pooled accumulators across all folds
all_y_true         = []
all_y_pred         = []
all_y_proba        = []
all_mouse_ids_pred = []   # mouse IDs for each test prediction
all_days_pred      = []   # days for each test prediction

print(f'Running LOGO CV ({logo.get_n_splits(groups=mouse_ids_all)} folds) ...')
print('=' * 60)

for fold, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, groups=mouse_ids_all)):
    test_mouse = np.unique(mouse_ids_all[test_idx])
    print(f'\nFold {fold+1}: test mouse = {test_mouse}, n_test_exams = {len(test_idx)}')

    X_train, y_train = X_all[train_idx], y_all[train_idx]
    X_test,  y_test  = X_all[test_idx],  y_all[test_idx]
    mouse_ids_test   = mouse_ids_all[test_idx]
    days_test        = days_all[test_idx]

    print(f'  Train set: {X_train.shape}  |  Test set: {X_test.shape}')
    print(f'  Train mice: {set(mouse_ids_all[train_idx])}  |  Test mice: {set(mouse_ids_test)}')

    # --- Build fresh model per fold (re-initialised from ImageNet weights) ---
    model = build_model()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )

    # --- Train on X_train only; validation_data for monitoring ONLY (no early stopping) ---
    history = model.fit(
        X_train, y_train,
        epochs=N_EPOCHS,
        batch_size=32,
        verbose=1,
        validation_data=(X_test, y_test),
    )

    # --- Predict ---
    y_pred_proba = model.predict(X_test).squeeze()
    y_pred       = (y_pred_proba > 0.5).astype(int)

    # --- Per-fold metrics ---
    acc_fold = accuracy_score(y_test, y_pred)
    try:
        auc_fold = roc_auc_score(y_test, y_pred_proba)
    except ValueError:
        auc_fold = float('nan')
    print(f'  Fold {fold+1} accuracy: {acc_fold:.4f}  AUC: {auc_fold:.4f}')

    # --- Save per-fold predictions CSV ---
    fold_dir = os.path.join(OUTPUT_DIR, f'fold_{fold+1}')
    os.makedirs(fold_dir, exist_ok=True)
    pd.DataFrame({
        'm_id':        mouse_ids_test,
        'day_of_study': days_test,
        'y_true':      y_test,
        'y_pred':      y_pred,
        'y_proba':     y_pred_proba,
    }).to_csv(os.path.join(fold_dir, 'predictions.csv'), index=False)

    # --- Accumulate ---
    all_y_true.extend(y_test.tolist())
    all_y_pred.extend(y_pred.tolist())
    all_y_proba.extend(y_pred_proba.tolist())
    all_mouse_ids_pred.extend(mouse_ids_test.tolist())
    all_days_pred.extend(days_test.tolist())

    # Free GPU memory between folds
    del model
    gc.collect()
    tf.keras.backend.clear_session()

print('\n' + '=' * 60)
print(f'All folds complete. Pooled exams: {len(all_y_true)}')

# Convert to arrays
all_y_true         = np.array(all_y_true)
all_y_pred         = np.array(all_y_pred)
all_y_proba        = np.array(all_y_proba)
all_mouse_ids_pred = np.array(all_mouse_ids_pred)
all_days_pred      = np.array(all_days_pred)

## A1 — Exam-Level Results

Pool all fold predictions and compute metrics at the exam level (each MRI scan is one observation).

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    """
    Compute classification metrics from pooled predictions.
    Uses labels=[0,1] in confusion_matrix to ensure 2x2 matrix even
    when one class is absent from predictions.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    spec = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    npv  = tn / (tn + fn) if (tn + fn) > 0 else float('nan')
    acc  = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else float('nan')

    try:
        auc_val = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc_val = float('nan')

    return {
        'AUC': auc_val,
        'Sensitivity': sens,
        'Specificity': spec,
        'PPV': ppv,
        'NPV': npv,
        'Accuracy': acc,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
    }


exam_metrics = compute_metrics(all_y_true, all_y_pred, all_y_proba)

print('Exam-Level Metrics (pooled across all LOGO folds):')
print('-' * 45)
for k, v in exam_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<14}: {v:.4f}')
    else:
        print(f'  {k:<14}: {v}')

# Save
exam_out = os.path.join(OUTPUT_DIR, 'summary_metrics_exam_level.csv')
pd.DataFrame([exam_metrics]).to_csv(exam_out, index=False)
print(f'\nSaved: {exam_out}')

## A2 — Animal-Level Results (Majority Vote)

Aggregate exam-level predictions to the mouse level using **majority vote** on `y_pred` across
all scans for that mouse. Probability is averaged. This addresses Reviewer Comment A2.

In [ ]:
# Build exam-level results DataFrame
results_df = pd.DataFrame({
    'mouse_id': all_mouse_ids_pred,
    'y_true':   all_y_true,
    'y_pred':   all_y_pred,
    'y_proba':  all_y_proba,
})

# Majority vote per mouse (A2)
animal_df = results_df.groupby('mouse_id').agg(
    y_true=('y_true', 'first'),
    y_pred_vote=('y_pred', lambda x: int(x.mode()[0])),
    y_proba_mean=('y_proba', 'mean'),
).reset_index()

print('Animal-Level Predictions (majority vote across exams):')
print(animal_df.to_string(index=False))
print()

# Animal-level metrics
if len(animal_df['y_true'].unique()) > 1:
    animal_metrics = compute_metrics(
        animal_df['y_true'].values,
        animal_df['y_pred_vote'].values,
        animal_df['y_proba_mean'].values,
    )
else:
    animal_metrics = {k: float('nan') for k in
                      ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Accuracy',
                       'TP', 'TN', 'FP', 'FN']}

print(f'Animal-Level Metrics (N={len(animal_df)} mice):')
print('-' * 40)
for k, v in animal_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<14}: {v:.4f}')
    else:
        print(f'  {k:<14}: {v}')

# Save
animal_df.to_csv(os.path.join(OUTPUT_DIR, 'summary_metrics_animal_level.csv'), index=False)
pd.DataFrame([animal_metrics]).to_csv(os.path.join(OUTPUT_DIR, 'animal_level_metrics.csv'), index=False)
print('\nSaved: summary_metrics_animal_level.csv, animal_level_metrics.csv')

## A2b — Animal-Level Results — Day-Weighted Vote

Alternative to majority vote: each exam's predicted probability is **weighted by its day of study**
before aggregating per animal. Two weighting schemes compared:
- **Linear:** weight = day_of_study (proportional to time)
- **Quadratic:** weight = day_of_study² (stronger late-bias)

In [ ]:
weighted_rows = []
for mouse in np.unique(all_mouse_ids_pred):
    mask   = all_mouse_ids_pred == mouse
    days   = all_days_pred[mask].astype(float)
    proba  = all_y_proba[mask]
    y_true = all_y_true[mask][0]

    w_lin     = days / days.sum()
    prob_lin  = float(np.dot(w_lin, proba))

    w_quad    = days**2 / (days**2).sum()
    prob_quad = float(np.dot(w_quad, proba))

    weighted_rows.append({
        'mouse_id':     mouse,
        'y_true':       int(y_true),
        'n_exams':      int(mask.sum()),
        'day_range':    f'{int(days.min())}–{int(days.max())}',
        'prob_uniform': float(proba.mean()),
        'pred_uniform': int(proba.mean() > 0.5),
        'prob_linear':  prob_lin,
        'pred_linear':  int(prob_lin > 0.5),
        'prob_quad':    prob_quad,
        'pred_quad':    int(prob_quad > 0.5),
    })

wv_df = pd.DataFrame(weighted_rows)
print('Per-mouse predictions (uniform vs linear vs quadratic weighting):')
print(wv_df.to_string(index=False))
print()

for scheme, pred_col, prob_col in [
    ('Uniform (majority vote)', 'pred_uniform', 'prob_uniform'),
    ('Linear day-weight',       'pred_linear',  'prob_linear'),
    ('Quadratic day-weight',    'pred_quad',    'prob_quad'),
]:
    m = compute_metrics(wv_df['y_true'].values, wv_df[pred_col].values, wv_df[prob_col].values)
    print(f'{scheme}:')
    print(f"  AUC={m['AUC']:.4f}  Sens={m['Sensitivity']:.4f}  "
          f"Spec={m['Specificity']:.4f}  PPV={m['PPV']:.4f}  "
          f"Acc={m['Accuracy']:.4f}  TP={m['TP']} FP={m['FP']} FN={m['FN']} TN={m['TN']}")
    print()

wv_df.to_csv(os.path.join(OUTPUT_DIR, 'animal_level_weighted_vote.csv'), index=False)
print('Saved: animal_level_weighted_vote.csv')

## A3 — Temporal Analysis

Split all pooled predictions into **early / mid / late** terciles based on `day_of_study`,
using the 33rd and 66th percentiles as thresholds. Report AUC, Sensitivity, Specificity per window.

Addresses Reviewer Comment A3: assess whether performance is consistent across the treatment timeline.

In [ ]:
t33, t66 = np.percentile(all_days_pred, [33, 66])
print(f'Tercile thresholds: early <= {t33:.0f}, mid <= {t66:.0f}, late > {t66:.0f} (days)')
print()

temporal_rows = []
for window_name, mask in [
    ('early', all_days_pred <= t33),
    ('mid',   (all_days_pred > t33) & (all_days_pred <= t66)),
    ('late',  all_days_pred > t66),
]:
    yt  = all_y_true[mask]
    yp  = all_y_pred[mask]
    ypr = all_y_proba[mask]
    n   = int(mask.sum())

    threshold_str = (f'<={t33:.0f}' if window_name == 'early' else
                     (f'<={t66:.0f}' if window_name == 'mid' else f'>{t66:.0f}'))

    if n == 0 or len(np.unique(yt)) < 2:
        row = {'window': window_name, 'n_exams': n, 'day_threshold': threshold_str,
               'AUC': float('nan'), 'Sensitivity': float('nan'), 'Specificity': float('nan')}
    else:
        m = compute_metrics(yt, yp, ypr)
        row = {'window': window_name, 'n_exams': n, 'day_threshold': threshold_str,
               'AUC': m['AUC'], 'Sensitivity': m['Sensitivity'], 'Specificity': m['Specificity']}

    temporal_rows.append(row)

temporal_df = pd.DataFrame(temporal_rows)
print('Temporal Analysis (early / mid / late terciles):')
print(temporal_df.to_string(index=False))

temporal_df.to_csv(os.path.join(OUTPUT_DIR, 'temporal_analysis.csv'), index=False)
print('\nSaved: temporal_analysis.csv')

## A4 — Bootstrap 95% Confidence Intervals

Compute 95% CIs for exam-level AUC, Sensitivity, Specificity, PPV, and Accuracy using
**non-parametric bootstrap** with 10,000 resamples (sampling with replacement from the pooled
exam-level predictions). Bootstrap samples where only one class is present are skipped.

Addresses Reviewer Comment A4: quantify uncertainty around reported point estimates.

In [ ]:
n_exams = len(all_y_true)
boot_metrics = defaultdict(list)

np.random.seed(SEED)
for _ in range(N_BOOTSTRAP):
    idx   = np.random.choice(n_exams, size=n_exams, replace=True)
    yt_b  = all_y_true[idx]
    yp_b  = all_y_pred[idx]
    ypr_b = all_y_proba[idx]

    if len(np.unique(yt_b)) < 2:
        continue

    m = compute_metrics(yt_b, yp_b, ypr_b)
    for key in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'Accuracy']:
        val = m[key]
        if not np.isnan(val):
            boot_metrics[key].append(val)

print(f'Bootstrap 95% CIs (N={N_BOOTSTRAP} resamples, exam level):')
print('-' * 60)

boot_rows = []
for metric in ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'Accuracy']:
    vals    = np.array(boot_metrics[metric])
    lo, hi  = np.percentile(vals, [2.5, 97.5])
    point   = exam_metrics[metric]
    n_valid = len(vals)
    boot_rows.append({
        'metric':             metric,
        'point_estimate':     point,
        'ci_lower_2_5':       lo,
        'ci_upper_97_5':      hi,
        'n_bootstrap_valid':  n_valid,
    })
    print(f'  {metric:<14}: {point:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  (n_valid={n_valid})')

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(os.path.join(OUTPUT_DIR, 'bootstrap_cis.csv'), index=False)
print(f'\nSaved: bootstrap_cis.csv')

print('\n' + '=' * 60)
print('All outputs saved to:', OUTPUT_DIR)
print('  - fold_*/predictions.csv  (per-fold exam predictions)')
print('  - summary_metrics_exam_level.csv')
print('  - summary_metrics_animal_level.csv')
print('  - animal_level_metrics.csv')
print('  - animal_level_weighted_vote.csv')
print('  - temporal_analysis.csv')
print('  - bootstrap_cis.csv')
print('')
print('Note: Feature importances not applicable for CNN (no symbolic feature importances).')
print('      For interpretability, see GradCAM analysis (gradcam.py).')